# RAG Project: BrewBean Coffee Co. Customer Assistant

This notebook builds a complete **Retrieval-Augmented Generation (RAG)** pipeline step by step.

**Business scenario (dummy data):** *BrewBean Coffee Co.* is a fictional coffee chain. We want an AI assistant that answers
customer questions (menu prices, refund policy, loyalty program, and so on) **using only the company's own documents**.

### What is RAG?
A Large Language Model (LLM) does not know our private company data. RAG solves this in two phases:

1. **Indexing (offline):** Load documents → split them into chunks → convert chunks into embeddings (vectors) → store them in a vector database.
2. **Retrieval + Generation (online):** User asks a question → find the most similar chunks → pass them to the LLM as context → LLM generates a grounded answer.

```
Documents ──► Loader ──► Text Splitter ──► Embeddings ──► Vector Store (FAISS)
                                                                 │
User Question ──► Embedding ──► Similarity Search ──► Top-k Chunks
                                                                 │
                                         Prompt (Context + Question) ──► LLM ──► Answer
```

### Tech stack
- **LangChain** for orchestration
- **OpenAI** for embeddings (`text-embedding-3-small`) and the chat model (`gpt-4o-mini`)
- **FAISS** as the vector store
- **python-dotenv** to load the API key safely from a `.env` file (never committed to GitHub)

## Step 1: Install the required libraries
Run this once. (You can skip it if you've already run `pip install -r requirements.txt`.)

In [ ]:
%pip install -q langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu python-dotenv

## Step 2: Load the API key securely
We **never hard-code API keys** in the notebook. Instead we create a `.env` file in the project folder:

```
OPENAI_API_KEY=sk-your-key-here
```

The `.env` file is listed in `.gitignore`, so it is never pushed to GitHub. A template, `.env.example`, is included in the repository.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads variables from the .env file into the environment

if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY loaded successfully")
else:
    raise ValueError("OPENAI_API_KEY not found. Create a .env file (see .env.example).")

## Step 3: Data loading
Our dummy company knowledge base is in the `data/` folder as plain-text files:

| File | Content |
|---|---|
| `company_overview.txt` | About the company, locations, contact |
| `menu_and_pricing.txt` | Drinks, bakery items, prices |
| `policies.txt` | Refund, delivery, opening hours |
| `loyalty_program.txt` | BeanPoints rewards program |
| `faq.txt` | Frequently asked questions |

We use LangChain's `DirectoryLoader` with `TextLoader` to load every `.txt` file as a `Document` object.
Each `Document` has `page_content` (the text) and `metadata` (for example, the source file name).

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()

print(f"Loaded {len(documents)} documents\n")
for doc in documents:
    print(f"- {doc.metadata['source']}  ({len(doc.page_content)} characters)")

In [ ]:
# Preview one document
print(documents[0].page_content[:500])

## Step 4: Split documents into chunks
LLMs and embedding models work best with small, focused pieces of text. Splitting into chunks means
retrieval returns only the relevant part of a document, not the whole file.

- `chunk_size=500`: maximum characters per chunk
- `chunk_overlap=50`: characters shared between neighbouring chunks, so a sentence cut at a boundary keeps its context

`RecursiveCharacterTextSplitter` tries to split on paragraphs first, then lines, then words, which keeps chunks meaningful.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks\n")
print("Example chunk:\n")
print(chunks[3].page_content)
print("\nMetadata:", chunks[3].metadata)

## Step 5: Create embeddings
An **embedding** is a list of numbers (a vector) that represents the meaning of a text.
Texts with similar meanings have vectors that are close together, which is what makes semantic search possible.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Quick demo: embed one sentence
sample_vector = embeddings.embed_query("What is the price of a cappuccino?")
print(f"Embedding dimension: {len(sample_vector)}")
print(f"First 5 values: {sample_vector[:5]}")

## Step 6: Indexing: store the chunks in a vector database (FAISS)
FAISS (Facebook AI Similarity Search) stores all chunk vectors and can quickly find the ones closest to a query vector.
We also **save the index to disk**, so next time we can load it without re-embedding (which saves time and API cost).

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local("faiss_index")

print(f"Indexed {vector_store.index.ntotal} vectors and saved them to ./faiss_index")

In [ ]:
# Later (or in another session) the saved index can be loaded like this:
vector_store = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,  # safe here because we created this index ourselves
)
print("Index loaded from disk")

## Step 7: Retrieval
Now we test retrieval on its own, **before** adding the LLM. Given a question, FAISS returns the `k` most similar chunks.
`similarity_search_with_score` also returns a distance score (lower = more similar).

In [ ]:
query = "What is your refund policy?"

results = vector_store.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results, start=1):
    print(f"--- Result {i} | score: {score:.4f} | source: {doc.metadata['source']} ---")
    print(doc.page_content[:300], "\n")

We wrap the vector store as a **retriever**, the standard LangChain interface that the RAG chain will use.

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

docs = retriever.invoke("How do BeanPoints work?")
for d in docs:
    print("•", d.metadata["source"], "→", d.page_content[:120].replace("\n", " "), "...")

## Step 8: Generation: build the RAG chain
We combine:
1. **Retriever**: fetches relevant chunks
2. **Prompt template**: tells the LLM to answer *only* from the given context
3. **LLM**: `gpt-4o-mini` with `temperature=0` for factual, consistent answers
4. **Output parser**: returns a plain string

This uses LangChain Expression Language (LCEL), where the `|` operator pipes each step into the next.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a helpful customer support assistant for BrewBean Coffee Co.
Answer the question using ONLY the context below.
If the answer is not in the context, say: "I'm sorry, I don't have that information. Please contact support@brewbean.example."

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG chain is ready")

## Step 9: Ask questions
Let's test the full pipeline with real customer-style questions.

In [ ]:
questions = [
    "How much does a Spanish Latte cost?",
    "Can I return coffee beans I ordered online?",
    "How many BeanPoints do I need for a free drink?",
    "What are your opening hours on public holidays?",
    "Do you have vegan options?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}\n")

### Testing a question that is *not* in our data
A good RAG system should not invent answers (*hallucinate*). The prompt tells the model to admit when it doesn't know.

In [ ]:
print(rag_chain.invoke("Do you sell pizza?"))

## Step 10: Show the sources (transparency)
In real applications, users trust answers more when they can see where the information came from.
This helper returns the answer **and** the source documents used.

In [ ]:
def ask_with_sources(question):
    source_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)
    print(f"Q: {question}\nA: {answer}\n\nSources:")
    for src in sorted({d.metadata["source"] for d in source_docs}):
        print("  -", src)

ask_with_sources("Is delivery free?")

## Step 11 (optional): Interactive chat
Run the cell and type questions. Type `exit` to stop.

In [ ]:
while True:
    user_q = input("You: ")
    if user_q.strip().lower() in {"exit", "quit"}:
        print("Goodbye!")
        break
    print("Assistant:", rag_chain.invoke(user_q), "\n")

## Summary

| Step | What we did | Tool |
|---|---|---|
| 1-2 | Installed libraries and loaded the API key securely from `.env` | `python-dotenv` |
| 3 | **Data loading**: read the company `.txt` files | `DirectoryLoader` |
| 4 | Split documents into overlapping chunks | `RecursiveCharacterTextSplitter` |
| 5 | Converted text into vectors | `OpenAIEmbeddings` |
| 6 | **Indexing**: stored the vectors in a vector DB and saved it to disk | `FAISS` |
| 7 | **Retrieval**: found the most relevant chunks for a question | `vector_store.as_retriever()` |
| 8-10 | **Generation**: answered questions grounded in the retrieved context, with sources | `ChatOpenAI` + LCEL |

**Security:** The API key lives only in `.env`, which `.gitignore` excludes, so no secrets are pushed to GitHub.